# MERRA-2 Trapping Variables

The original weather dataset has four variables — temperature, humidity, and the two wind
components. None of them measure the thing that actually traps pollution over the LA basin.

Los Angeles sits in a bowl under a persistent subsidence inversion. Warm air aloft caps the
cooler marine air below, and everything emitted underneath that cap stays there. Temperature
and wind are *correlated* with this, which is why the models work at all, but they never
observe it directly.

Four new variables, all subsets of MERRA-2 products already used by this project:

| variable | product | what it gives us |
|---|---|---|
| `PBLH` | M2T1NXFLX | planetary boundary layer height — the depth of air pollution can mix into |
| `T850` | M2T1NXSLV | temperature at 850 hPa; against surface temperature this gives inversion strength |
| `SLP` | M2T1NXSLV | sea-level pressure — high pressure means subsidence and stagnation |
| `TQV` | M2T1NXSLV | total column water vapor |

From these come the two features that matter most physically:

- **Inversion strength** = `T850 − T2M`. Positive means warmer air sits above cooler air — a
  temperature inversion, the cap.
- **Ventilation index** = `PBLH × wind speed`. The standard operational air-quality metric: how
  much air volume is available per unit time to dilute emissions. Low ventilation is the
  classic smog setup.

Output: `data/processed/la_daily_trapping_2016_2025.csv`

## Setup

In [ ]:
import glob, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

FLX       = ROOT / 'data' / 'raw' / 'merra2_flx'
SLVX      = ROOT / 'data' / 'raw' / 'merra2_slv_extra'
PROCESSED = ROOT / 'data' / 'processed'
OUT_CSV   = PROCESSED / 'la_daily_trapping_2016_2025.csv'

for d, what in [(FLX, 'PBLH granules'), (SLVX, 'T850/SLP/TQV granules')]:
    assert d.exists(), (f'{what} missing at {d}. These are gitignored (~150 MB) — '
                        'see the README for the download command.')

flx_files  = sorted(glob.glob(str(FLX  / '*.nc4')))
slvx_files = sorted(glob.glob(str(SLVX / '*.nc4')))
print(f'PBLH granules:        {len(flx_files)}')
print(f'T850/SLP/TQV granules: {len(slvx_files)}')

## Aggregating hourly to daily

Same convention as the main weather pipeline: spatial mean over the 3×3 grid to get one value
per hour, then daily statistics over the 24 hours.

`PBLH` gets special treatment. Its daily **minimum** matters more than its mean — the shallowest
the mixing layer gets is what determines how concentrated pollution becomes overnight and in the
early morning. The night-time mean is tracked separately for the same reason.

In [ ]:
def summarize_flx(path):
    with xr.open_dataset(path) as ds:
        pblh = ds.PBLH.mean(dim=('lat', 'lon')).values
        date = pd.Timestamp(ds.time.values[0]).normalize()
        hours = pd.DatetimeIndex(ds.time.values).hour
    night = pblh[(hours < 7) | (hours >= 20)]
    return {'date': date,
            'pblh_mean':  pblh.mean(),
            'pblh_min':   pblh.min(),
            'pblh_max':   pblh.max(),
            'pblh_night': night.mean()}


def summarize_slvx(path):
    with xr.open_dataset(path) as ds:
        t850 = ds.T850.mean(dim=('lat', 'lon')).values - 273.15
        slp  = ds.SLP.mean(dim=('lat', 'lon')).values / 100.0    # Pa -> hPa
        tqv  = ds.TQV.mean(dim=('lat', 'lon')).values
        date = pd.Timestamp(ds.time.values[0]).normalize()
    return {'date': date,
            't850_mean': t850.mean(), 't850_max': t850.max(),
            'slp_mean': slp.mean(), 'slp_max': slp.max(),
            'tqv_mean': tqv.mean()}


def run(files, fn, label):
    rows, failures = [], []
    t0 = time.time()
    for i, f in enumerate(files, 1):
        try:
            rows.append(fn(f))
        except Exception as e:
            failures.append((Path(f).name, repr(e)))
        if i % 750 == 0 or i == len(files):
            el = time.time() - t0
            print(f'  {label} {i:5}/{len(files)}  {el:5.1f}s  ~{el/i*(len(files)-i):4.1f}s left  '
                  f'({len(failures)} failures)')
    if failures:
        print(f'  {len(failures)} failures, first few:')
        for name, err in failures[:5]:
            print('   ', name, '->', err)
    return pd.DataFrame(rows)


flx_df  = run(flx_files,  summarize_flx,  'PBLH')
slvx_df = run(slvx_files, summarize_slvx, 'SLVX')
print(f'\nPBLH rows {len(flx_df)}, SLVX rows {len(slvx_df)}')

## Joining to the existing weather series

The derived features need surface temperature and wind speed from the original pipeline, so
everything is merged on date. An inner join keeps only days where all three sources agree,
which is the honest thing to do — a partially-populated row would silently poison the lags
downstream.

In [ ]:
weather = pd.read_csv(PROCESSED / 'la_daily_weather_2016_2025.csv', parse_dates=['date'])

trap = (weather[['date', 't2m_mean', 't2m_max', 'wind_speed_mean']]
        .merge(flx_df,  on='date', how='inner')
        .merge(slvx_df, on='date', how='inner'))

print(f'weather {len(weather)} | PBLH {len(flx_df)} | SLVX {len(slvx_df)} -> joined {len(trap)}')
print(f'range: {trap.date.min().date()} -> {trap.date.max().date()}')

full_range = pd.date_range(trap.date.min(), trap.date.max(), freq='D')
gaps = full_range.difference(trap.date)
print(f'gaps inside the joined range: {len(gaps)}')
if len(gaps):
    print('  first few:', [str(d.date()) for d in gaps[:8]])

## The derived physics

Two features carry most of the intended signal:

- **`inversion_strength` = T850 − T2M.** Positive means air at 850 hPa (roughly 1.5 km up) is
  *warmer* than the surface — a capping inversion. The stronger it is, the more firmly pollution
  is held down.
- **`ventilation_index` = PBLH × wind speed.** Mixing depth times transport speed. This is the
  operational metric forecasters actually use; low values are the smog setup.

`pblh_range` is added as a proxy for how vigorously the boundary layer breathes over the day.

In [ ]:
trap['inversion_strength']     = trap['t850_mean'] - trap['t2m_mean']
trap['inversion_strength_max'] = trap['t850_max']  - trap['t2m_max']
trap['ventilation_index']      = trap['pblh_mean'] * trap['wind_speed_mean']
trap['ventilation_index_min']  = trap['pblh_min']  * trap['wind_speed_mean']
trap['pblh_range']             = trap['pblh_max']  - trap['pblh_min']

NEW_COLS = ['pblh_mean', 'pblh_min', 'pblh_max', 'pblh_night', 'pblh_range',
            't850_mean', 'slp_mean', 'tqv_mean',
            'inversion_strength', 'inversion_strength_max',
            'ventilation_index', 'ventilation_index_min']

print(trap[NEW_COLS].describe().T.round(2).to_string())
print(f'\ndays with a capping inversion (T850 > T2M): '
      f'{(trap.inversion_strength > 0).mean():.1%}')

## Do these actually track AQI?

The whole justification for this work is that these variables measure the trapping mechanism
directly. If they do, they should correlate with AQI at least as strongly as the temperature
features already in the model.

In [ ]:
aqi = pd.read_csv(PROCESSED / 'la_daily_aqi_5pollutants_v2_2016_2025.csv',
                  parse_dates=['date'])[['date', 'daily_aqi', 'dominant_pollutant']]
chk = trap.merge(aqi, on='date', how='inner')

corr = chk[NEW_COLS + ['daily_aqi']].corr()['daily_aqi'].drop('daily_aqi')
print('Correlation with daily AQI (new variables):')
print(corr.sort_values(key=abs, ascending=False).round(3).to_string())

print('\nFor comparison, the strongest existing feature:')
base = pd.read_csv(PROCESSED / 'la_modeling_dataset.csv', parse_dates=['date'])
print(f"  t2m_max  {base[['t2m_max','daily_aqi']].corr().iloc[0,1]:.3f}")

print('\nSplit by which pollutant set the daily max:')
for p in ['Ozone', 'PM2.5']:
    sub = chk[chk.dominant_pollutant == p]
    c = sub[NEW_COLS + ['daily_aqi']].corr()['daily_aqi'].drop('daily_aqi')
    top = c.sort_values(key=abs, ascending=False).head(4)
    print(f'  {p} (n={len(sub)}): ' + ', '.join(f'{k} {v:+.3f}' for k, v in top.items()))

## Saving

In [ ]:
out = trap[['date'] + NEW_COLS].copy()
out.to_csv(OUT_CSV, index=False)
print(f'Saved {OUT_CSV.relative_to(ROOT)}  ({len(out)} rows, {out.shape[1]} columns)')
out.head()